# Earth–Moon L1 northern halo family near the bifurcation

Generate and validate a short halo family with the existing targeting and continuation routines.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from tno_dynamics.continuation import grow_halo_family
from tno_dynamics.dynamics.cr3bp import jacobi_constant
from tno_dynamics.plotting import plot_halo_family_3d
from tno_dynamics.propagation import propagate_cr3bp
from tno_dynamics.targeting import correct_halo_orbit

## Near-bifurcation seeds

The displayed values are rounded, so the state is treated as an initial guess and corrected before continuation.

In [ ]:
mu = 0.01215058560962404

seed_guess = np.array(
    [
        0.823390678,
        0.0,
        0.001659057,
        0.0,
        0.126372300,
        0.0,
    ]
)

first_state, first_half_period, first_residuals = correct_halo_orbit(
    seed_guess,
    mu,
)

In [ ]:
second_guess = first_state.copy()
second_guess[2] += 2e-4

second_state, second_half_period, second_residuals = correct_halo_orbit(
    second_guess,
    mu,
)

print("First corrected state:", first_state)
print(f"First full period: {2.0 * first_half_period:.16e}")
print(f"First final residual: {first_residuals[-1]:.6e}")
print("Second corrected state:", second_state)
print(f"Second full period: {2.0 * second_half_period:.16e}")
print(f"Second final residual: {second_residuals[-1]:.6e}")

## Fixed-step pseudo-arclength continuation

In [ ]:
family_states, half_periods, continuation_histories = grow_halo_family(
    first_state,
    second_state,
    first_half_period,
    second_half_period,
    mu,
    number_of_orbits=25,
    step_size=2e-4,
)

periods = 2.0 * half_periods

In [ ]:
for index, (state, period) in enumerate(zip(family_states, periods, strict=True)):
    print(
        f"{index:02d}  x0={state[0]: .9f}  z0={state[2]: .9f}  "
        f"ydot0={state[4]: .9f}  period={period:.9f}"
    )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(family_states[:, 2], family_states[:, 0], marker="o", ms=3)
axes[0].set(xlabel="$z_0$", ylabel="$x_0$", title="Initial position")
axes[1].plot(family_states[:, 2], family_states[:, 4], marker="o", ms=3)
axes[1].set(xlabel="$z_0$", ylabel=r"$\dot{y}_0$", title="Initial velocity")
axes[2].plot(family_states[:, 2], periods, marker="o", ms=3)
axes[2].set(xlabel="$z_0$", ylabel="Full period", title="Period")
for ax in axes:
    ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## Full-period validation

In [ ]:
trajectories = []
closure_norms = []
jacobi_drifts = []

for state, period in zip(family_states, periods, strict=True):
    t_eval = np.linspace(0.0, period, 1001)
    solution = propagate_cr3bp(state, (0.0, period), mu, t_eval=t_eval)
    trajectories.append(solution)
    closure_norms.append(np.linalg.norm(solution.y[:, -1] - state))

    jacobi_values = np.array(
        [jacobi_constant(sample, mu) for sample in solution.y.T]
    )
    jacobi_drifts.append(np.max(np.abs(jacobi_values - jacobi_values[0])))

closure_norms = np.asarray(closure_norms)
jacobi_drifts = np.asarray(jacobi_drifts)

In [ ]:
fig, ax = plot_halo_family_3d(
    family_states,
    periods,
    trajectories,
    mu,
    l1_x=0.836915125772357,
)
plt.show()

In [ ]:
family_indices = np.arange(len(family_states))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(family_indices, closure_norms, marker="o", ms=3)
axes[0].set(xlabel="Family index", ylabel="Closure norm", title="Closure error")
axes[1].semilogy(family_indices, jacobi_drifts, marker="o", ms=3)
axes[1].set(
    xlabel="Family index",
    ylabel="Maximum Jacobi drift",
    title="Jacobi conservation",
)
for ax in axes:
    ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
plt.show()